# Phase 5-B1｜Trusted Approval & Authorization Binding — Security Remediation Acceptance

**Notebook 版本：`PHASE5B1-APPROVAL-v2`**　｜　Code pin `eb72cd0`　｜　分支 `feature/phase5b1-trusted-approval`

`Proposal → Pending Request → Human Review → Verified Approval → Authorization Binding → Governed Request → Orchestrator`

- **隔離驗收（自動）**：TEST_FIXTURE 核准證據與 SYNTHETIC_FIXTURE 綁定環境。走真實的 verifier、binding、governance、orchestrator 程式，並跑 18 項威脅負面測試。
- **真實 GitHub 核對（唯讀）**：研究負責人**本人**在 GitHub 網頁留言，貼上第 5 格印出的核准行；notebook 用 live GitHub API 驗證作者身分、repo、內容，再對真實帳本做 **dry-run** 綁定。預期因為沒有 A2 provenance、授權 1/1 已用完而被拒；不寫入任何東西。
- **不執行回測、不消耗額度、不讀 Holdout、不建立授權、不呼叫券商 API。** GitHub token 只從 Colab Secrets 讀取，不印出、不寫檔。
- **安全修復驗收（v2）**：Codex 審查 PR #12（`cf15654`）判定 NO_GO。本 notebook 會在 pinned checkout 重新執行全部漏洞重現案例，**每一案都必須是 BLOCKED**（gate B1-SEC）。
- 舊的 Colab GO 證據（`approvals/20261107_173254_a3251d3`）保留不動，作為修復前版本；它的安全判定已被推翻。
- 輸出：`MyDrive/Quant_Research/strategy-research-agent/approvals/<YYYYMMDD_HHMMSS>_<pin7>/`。共 8 個檔案，最後寫入的是 `ARCHIVE_INDEX.json`（涵蓋所有檔案的 hash），不可覆寫。

**請在全新的 Colab runtime 從 GitHub 開啟並 Run all。**

In [ ]:
# --- 1. 設定（不要修改 pin）---
NOTEBOOK_VERSION = "PHASE5B1-APPROVAL-v2"
NOTEBOOK_PATH    = "notebooks/Phase5B1_Approval_Binding_Acceptance.ipynb"
AGENT_REPO = "https://github.com/hh4832/strategy-research-agent.git"
BRANCH     = "feature/phase5b1-trusted-approval"
PIN_SHA    = "eb72cd02770793a168b704af94824a7300e2010d"
DRIVE_BASE = "/content/drive/MyDrive/Quant_Research/strategy-research-agent"
AGENT_ROOT = "/content/strategy-research-agent-phase5b1"
PROBE_REQUEST = "docs/examples/phase5b1/PROBE_PENDING_tsm_sma520_research_only_real.json"
REAL_AUTH = "RA-IBKR_TSM-SMA520_DEV-18350b3adba8e28a7e5e"

import json, os, re, subprocess, sys
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo
assert sys.version_info >= (3, 11), sys.version
CLEAN_RUNTIME = not any(m in sys.modules for m in ("strategy_agent", "market_research_core"))
AGENT_ROOT = Path(AGENT_ROOT)
def git(*a, cwd=None):
    r = subprocess.run(["git", *a], cwd=cwd, text=True, capture_output=True)
    if r.returncode: raise RuntimeError("git " + a[0] + " failed: " + r.stderr[-400:])
    return r.stdout.strip()
print(NOTEBOOK_VERSION, "| pin", PIN_SHA[:7], "| python", sys.version.split()[0], "| clean runtime:", CLEAN_RUNTIME)

In [ ]:
# --- 2. Version Sync ---
assert CLEAN_RUNTIME, "STOP: 此 runtime 已載入 strategy_agent / market_research_core；請 Runtime → Restart session 後 Run all"
if not AGENT_ROOT.exists():
    git("clone", "--quiet", AGENT_REPO, str(AGENT_ROOT)); git("checkout", "--quiet", "--detach", PIN_SHA, cwd=AGENT_ROOT)
    print("Fresh checkout created")
else:
    print("Existing checkout detected: verifying before reuse")
remote = git("remote", "get-url", "origin", cwd=AGENT_ROOT)
assert remote.rstrip("/").removesuffix(".git") == AGENT_REPO.removesuffix(".git"), "STOP: wrong remote " + remote
head = git("rev-parse", "HEAD", cwd=AGENT_ROOT)
status = git("status", "--porcelain", "--untracked-files=all", cwd=AGENT_ROOT)
git("fetch", "--quiet", "origin", BRANCH, cwd=AGENT_ROOT)
remote_head = git("rev-parse", "FETCH_HEAD", cwd=AGENT_ROOT)
on_branch = subprocess.run(["git", "merge-base", "--is-ancestor", PIN_SHA, remote_head], cwd=AGENT_ROOT).returncode == 0
nb = subprocess.run(["git", "show", f"{remote_head}:{NOTEBOOK_PATH}"], cwd=AGENT_ROOT, text=True, capture_output=True)
gh_version = (re.search(r'NOTEBOOK_VERSION = \\"([^\\"]+)\\"', nb.stdout) or [None, None])[1] if nb.returncode == 0 else None
after_pin = git("diff", "--name-only", PIN_SHA, remote_head, cwd=AGENT_ROOT).splitlines() if remote_head != PIN_SHA else []
SYNC = {"notebook_version": NOTEBOOK_VERSION, "notebook_version_on_github_head": gh_version, "github_remote_head": remote_head,
        "checkout_head": head, "pinned_sha": PIN_SHA, "pin_on_branch": on_branch, "working_tree_clean": status == "",
        "files_changed_after_pin": after_pin}
print(json.dumps(SYNC, indent=2))
assert head == PIN_SHA, "STOP: checkout HEAD ≠ pin（不自動 checkout/reset；請換 AGENT_ROOT 或重啟 runtime）"
assert status == "", "STOP: checkout 有變更"
assert on_branch, "STOP: pin 不在 GitHub 分支上"
assert gh_version in (None, NOTEBOOK_VERSION), f"STOP: 這是舊 notebook（{NOTEBOOK_VERSION}），GitHub 上是 {gh_version}；請從 GitHub 重新開啟"
code_after_pin = [p for p in after_pin if not p.startswith(("docs/", "notebooks/"))]
assert not code_after_pin, f"STOP: pin 之後 GitHub 上有程式／治理變更 {code_after_pin}；需要新的 pin"
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(AGENT_ROOT) + "[test]"], check=True)
print("VERSION SYNC: PASS")

In [ ]:
# --- 3. GitHub token：只從 Colab Secrets（左側鑰匙圖示，名稱 GH_TOKEN；建議 fine-grained、只讀此 repo 的 Issues/PR）---
TOKEN_OK = False
try:
    from google.colab import userdata
    _t = userdata.get("GH_TOKEN")
    if _t:
        os.environ["GH_TOKEN"] = _t; TOKEN_OK = True
    del _t
except Exception as e:                       # noqa: BLE001
    print("Colab Secret GH_TOKEN unavailable:", type(e).__name__)
print({"github_token_present": TOKEN_OK, "printed": False, "written_to_file": False})

In [ ]:
# --- 4. Regression tests（subprocess；Core 不安裝 → Core 測試依設計 skip）---
def py(args, **kw):
    return subprocess.run([sys.executable, *args], cwd=AGENT_ROOT, text=True, capture_output=True, **kw)
env_no_token = {k: v for k, v in os.environ.items() if k not in ("GH_TOKEN", "GITHUB_TOKEN")}
t = subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=AGENT_ROOT, text=True, capture_output=True, timeout=3600,
                   env=env_no_token)
TESTS = {"returncode": t.returncode, "summary": t.stdout.strip().splitlines()[-1] if t.stdout.strip() else "",
         "python": sys.version.split()[0], "where": "Colab"}
print(t.stdout[-1500:]); print(TESTS)
assert t.returncode == 0, "STOP: regression failed"
assert git("status", "--porcelain", "--untracked-files=all", cwd=AGENT_ROOT) == "", "STOP: tests modified the checkout"

In [ ]:
# --- 5. 真實 GitHub 核准行（唯讀核對用；必須由你本人在 GitHub 網頁貼上，不可交給 AI agent 代發）---
r = py(["-m", "strategy_agent.binding", "approval-line", "--request", PROBE_REQUEST, "--intent", "BIND_AUTHORIZATION",
        "--authorization", REAL_AUTH, "--hours", "48"])
assert r.returncode == 0, r.stderr[-1000:]
LINE = json.loads(r.stdout)["post_this_line_yourself"]
print("請到 Phase 5-B1 的 Draft PR，用你自己的帳號新增一則留言，內容包含下面這一行（可另外加說明文字）：\n")
print(LINE)
print("\n這是 PROBE：它指向已用完 1/1 的授權與沒有 A2 provenance 的探針請求，dry-run 綁定一定會被拒，不會寫入任何東西。")

In [ ]:
# --- 6. 貼上你那則留言的網址（留空 = 真實核對 PENDING；只做隔離驗收）---
EVIDENCE_REF = ""
assert EVIDENCE_REF == "" or EVIDENCE_REF.startswith("https://github.com/hh4832/strategy-research-agent/"), "網址格式不對"
print("real GitHub check:", "WILL RUN (read-only)" if EVIDENCE_REF and TOKEN_OK else "PENDING")

In [ ]:
# --- 7. 漏洞重現（pinned checkout）+ Drive + B1-01..B1-10 / B1-SEC acceptance（全新 subprocess）---
rem_out = Path("/content/b1_remediation.json")
rr = py(["docs/evidence/phase5b1_remediation/reproduce.py", str(AGENT_ROOT), str(rem_out)], timeout=1800, env=env_no_token)
print(rr.stdout[-3000:]); assert rr.returncode == 0, rr.stderr[-2000:]
assert git("status", "--porcelain", "--untracked-files=all", cwd=AGENT_ROOT) == "", "STOP: reproduction modified the checkout"
CLOUD = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    CLOUD = Path(DRIVE_BASE).is_dir()
except Exception as e:                       # noqa: BLE001
    print("Drive unavailable:", type(e).__name__, e)
OUT_BASE = Path(DRIVE_BASE) / "approvals" if CLOUD else Path("/content/approvals_LOCAL_CLOUD_PENDING")
OUT = OUT_BASE / f"{datetime.now(ZoneInfo('Asia/Taipei')):%Y%m%d_%H%M%S}_{PIN_SHA[:7]}"
assert not OUT.exists(), "STOP: output exists (never overwritten)"
tj = Path("/content/b1_tests.json"); tj.write_text(json.dumps(TESTS))
cj = Path("/content/b1_context.json"); cj.write_text(json.dumps({"notebook": NOTEBOOK_VERSION, "version_sync": SYNC,
    "clean_colab_runtime": CLEAN_RUNTIME, "cloud": CLOUD, "real_check_requested": bool(EVIDENCE_REF and TOKEN_OK),
    "evidence_ref": EVIDENCE_REF or None, "python": sys.version.split()[0], "tests": TESTS,
    "superseded_evidence": "approvals/20261107_173254_a3251d3 (pre-remediation GO, overturned by Codex security review)"}))
cmd = ["-m", "strategy_agent.binding.acceptance", "--out-dir", str(OUT), "--tests-json", str(tj), "--context-json", str(cj),
       "--remediation-json", str(rem_out)] + (["--cloud"] if CLOUD else [])
if EVIDENCE_REF and TOKEN_OK:
    cmd += ["--evidence-ref", EVIDENCE_REF]
r = py(cmd, timeout=3600)
print(r.stdout[-5000:]); print(r.stderr[-2000:])
RES = json.loads(r.stdout[r.stdout.index("{"):])
print("VERDICT:", RES["verdict"])
assert git("status", "--porcelain", "--untracked-files=all", cwd=AGENT_ROOT) == "", "STOP: acceptance modified the checkout"

In [ ]:
# --- 8. 封存驗證（ARCHIVE_INDEX 涵蓋全部檔案；之後不再寫入）+ token 掃描 + 下載 ---
import zipfile
v = py(["-m", "strategy_agent.binding.acceptance", "--out-dir", str(OUT), "--verify-archive"])
print(v.stdout); assert v.returncode == 0, "STOP: archive index verification failed"
tok = os.environ.get("GH_TOKEN", "")
files = sorted(p for p in OUT.iterdir() if p.is_file())
assert not tok or not any(tok in p.read_text(errors="ignore") for p in files), "STOP: token found in output"
print((OUT / "run_info.txt").read_text())
zp = Path("/content") / (OUT.name + "_phase5b1_approval.zip")
with zipfile.ZipFile(zp, "w") as z:
    for p in files:
        z.write(p, p.name)
try:
    from google.colab import files as F; F.download(str(zp))
except Exception as e:                       # noqa: BLE001
    print("download skipped:", e, "| zip:", zp)